# Master verification — Google Colab (standalone)

This notebook is **self-contained for Colab**: it clones your GitHub repo, installs `requirements.txt`, puts `src/` on `PYTHONPATH`, then runs the same **PubMedQA adaptive shootout** as `MasterVerification.TEMPLATE.ipynb` (100 questions, `seed=42`).

**You need**

1. A **GPU** runtime (T4 / L4 / A100).
2. Colab secret **`HF_TOKEN`** (Hugging Face) with access to `meta-llama/Meta-Llama-3.1-8B-Instruct`.
3. **`REPO_URL`** in the next cell pointing at a branch that contains this code (`src/adaptive_rag/`, `requirements.txt`).
4. **Artifacts** on the clone machine: either commit small smoke artifacts (not recommended for large weights) or run pipeline cells below once, **or** mount Drive and set `ADAPTIVE_RAG_ARTIFACTS` / `ADAPTIVE_RAG_DATA` in the optional cell.

If this notebook completes, the same logic in `src/` and `scripts/` is what executed — good sanity check for the Python layout.


In [ ]:
# --- EDIT THIS ---
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # must contain src/ + requirements.txt
BRANCH = "main"
CLONE_DIR = "/content/adaptive_rag_project"
# -----------------


In [ ]:
import shutil
import subprocess
from pathlib import Path

p = Path(CLONE_DIR)
if p.exists():
    shutil.rmtree(p)
subprocess.run(
    ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(p)],
    check=True,
)
print("Cloned to", p)


In [ ]:
# Colab: move into repo (magic works in Colab / Jupyter)
%cd /content/adaptive_rag_project


In [ ]:
import subprocess, sys
from pathlib import Path

root = Path(CLONE_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip"],
    cwd=str(root),
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")],
    cwd=str(root),
    check=True,
)
print("pip install done")


In [ ]:
import os, sys
from pathlib import Path

ROOT = Path(CLONE_DIR).resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("CWD:", os.getcwd())
print("sys.path[0]:", sys.path[0])


In [ ]:
import os

try:
    from google.colab import userdata

    tok = userdata.get("HF_TOKEN")
except Exception:
    tok = os.environ.get("HF_TOKEN")

from huggingface_hub import login

if not tok:
    raise RuntimeError("Set Colab secret HF_TOKEN or export HF_TOKEN before running.")
login(token=tok)
print("HF login OK")


### Optional: artifacts on Google Drive

If `data/` and `artifacts/` are **not** inside the cloned repo (typical: large FAISS + LoRA live on Drive), **uncomment** the next cell and fix the paths. `adaptive_rag.config` reads `ADAPTIVE_RAG_ARTIFACTS` and `ADAPTIVE_RAG_DATA`.


In [ ]:
# Optional — uncomment and edit paths after `drive.mount`
# from google.colab import drive
# drive.mount("/content/drive")
# import os
# os.environ["ADAPTIVE_RAG_ARTIFACTS"] = "/content/drive/MyDrive/YourFolder/artifacts"
# os.environ["ADAPTIVE_RAG_DATA"] = "/content/drive/MyDrive/YourFolder/data"
# print(os.environ.get("ADAPTIVE_RAG_ARTIFACTS"), os.environ.get("ADAPTIVE_RAG_DATA"))
pass


### Optional: build artifacts inside this session

If you have **no** CSV / FAISS / LoRA yet, run the pipeline once (long, needs GPU + HF). Comment out if you already have files under `data/` and `artifacts/`.


In [ ]:
# Set True to run the full pipeline in Colab (very long). Requires HF_TOKEN and a strong GPU.
RUN_PIPE = False

import subprocess
import sys
from pathlib import Path

ROOT = Path(CLONE_DIR)
if RUN_PIPE:
    steps = [
        [sys.executable, "scripts/02_generate_medhallu_router_csv.py", "--batch-size", "8"],
        [sys.executable, "scripts/03_train_lora_router.py", "--epochs", "2"],
        [sys.executable, "scripts/04_build_pubmed_faiss.py"],
        [sys.executable, "scripts/05_train_baseline_routers.py", "--skip-bert"],
    ]
    for cmd in steps:
        print(">>", " ".join(cmd))
        subprocess.run(cmd, cwd=str(ROOT), check=True)
else:
    print("Skipping pipeline (RUN_PIPE=False). Upload data/ + artifacts/ or mount Drive.")


In [ ]:
import joblib
import pandas as pd

from adaptive_rag.config import (
    ARTIFACTS_DIR,
    DISTILBERT_ROUTER_DIR,
    LORA_ROUTER_DIR,
    MEDHALLU_TRAINING_CSV,
    PUBMED_FAISS_INDEX,
    PUBMED_FAISS_MAPPING,
)
from adaptive_rag.router_baselines import train_xgboost_router

print("MEDHALLU_TRAINING_CSV:", MEDHALLU_TRAINING_CSV, "exists:", MEDHALLU_TRAINING_CSV.exists())
print("FAISS:", PUBMED_FAISS_INDEX.exists(), PUBMED_FAISS_MAPPING.exists())
print("LoRA:", LORA_ROUTER_DIR.exists())

xgb_path = ARTIFACTS_DIR / "xgb_router.joblib"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

assert MEDHALLU_TRAINING_CSV.exists(), "Missing CSV. Run script 02, set ADAPTIVE_RAG_DATA, or upload data/."
assert PUBMED_FAISS_INDEX.exists() and PUBMED_FAISS_MAPPING.exists(), "Missing FAISS. Run script 04 or set ADAPTIVE_RAG_ARTIFACTS."
assert LORA_ROUTER_DIR.exists(), "Missing LoRA adapter. Run script 03 or set ADAPTIVE_RAG_ARTIFACTS."

if xgb_path.exists():
    xgb_model = joblib.load(xgb_path)
    print("Loaded", xgb_path)
else:
    df = pd.read_csv(MEDHALLU_TRAINING_CSV, engine="python", on_bad_lines="skip")
    xgb_model, _ = train_xgboost_router(df["prompt"].tolist(), df["label"].tolist())
    joblib.dump(xgb_model, xgb_path)
    print("Trained and saved", xgb_path)

distil_dir = DISTILBERT_ROUTER_DIR if DISTILBERT_ROUTER_DIR.exists() else None
if distil_dir is None:
    print("DistilBERT router not found; shootout will skip D_BERT")


In [ ]:
from adaptive_rag.report_benchmark import run_shootout_report
from adaptive_rag.adaptive_shootout import print_summary

summary, df = run_shootout_report(
    faiss_index_path=PUBMED_FAISS_INDEX,
    faiss_mapping_path=PUBMED_FAISS_MAPPING,
    lora_adapter_dir=LORA_ROUTER_DIR,
    distilbert_dir=distil_dir,
    xgb_model=xgb_model,
    n_samples=100,
    seed=42,
)
print_summary(summary)
df
